In [1]:
# !pip install spacy
# !python -m spacy download en_core_web_sm
# !pip install scispacy
# !pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.4.0/en_core_sci_sm-0.4.0.tar.gz


In [ ]:
from helpers import *

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForTokenClassification, AdamW
from tqdm import tqdm
import re
from pypdf import PdfReader, PdfWriter
import pandas as pd
import json
import spacy

## Step 1: Generate artificially filled forms

#### Commercial Prescription drug forms

In [3]:
form_generator = FormGenerator()
input_pdf_path = 'ClaimForms\commercial-prescription-drug-claim-form-english.pdf'

print("Generating forms...")
for i in range(50):
    dict = form_generator.field_values_to_fill_cpdcf(input_pdf_path)
    form_generator.write_form(input_pdf_path, f'generated_forms\cpdcf_{i}.pdf', 0, dict)

print("Form generation complete.")

Generating forms...
Form generation complete.


#### Medical benefit form

In [2]:
form_generator = FormGenerator()
input_pdf_path = 'ClaimForms\medical-benefits-claim-form.pdf'

print("Generating forms...")
for i in range(70,100):
    dict = form_generator.field_values_to_fill_mbr(input_pdf_path)
    form_generator.write_form(input_pdf_path, f'generated_forms\mbr_{i}.pdf', 1, dict)

print("Form generation complete.")

Generating forms...
Form generation complete.


## Step 2: Train on generated forms to extract information
### Training BERT for NER to extract important information from forms

### 2.1 Annotate data

In [ ]:
annotator = annotate_data()

## Annotate for Prescripotion drug forms
for i in range(50):

    input_pdf_path = f"generated_forms\cpdcf_{i}.pdf"
    input_text, annotation = annotator.create_cpdcf_df(input_pdf_path)

    final_df = final_df.append({'input_text': input_text, 'annotation': annotation}, ignore_index=True)

## Annotate for Medical Benefits forms
for i in range(50):

    input_pdf_path = f"generated_forms\mbr_{i}.pdf"
    input_text, annotation = annotator.create_mbr_df(input_pdf_path)

    final_df = final_df.append({'input_text': input_text, 'annotation': annotation}, ignore_index=True)

final_df.to_csv('training_dataset.csv', index=False)



In [11]:
df = pd.read_csv('training_dataset.csv')
training_data = [
    (row['input_text'], ast.literal_eval(row['annotation']))
    for _, row in df.iterrows()
]
training_data[0]

('Last Name, First, Middle Initial: Jennifer Davis, Patient Birthdate (MM/DD/YYYY): 11/04/1962, Gender: /female, Employee, Spouse, Dependent: /dependent, Aetna Member Number (claim cannot be processed without number): 01082258402348',
 {'entities': [(33, 46, 'PATIENT_NAME'),
   (74, 84, 'PATIENT_BIRTHDATE'),
   (94, 100, 'GENDER'),
   (204, 216, 'AETNA_ID')]})

### 2.2 Prepare data for training

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

label_map = {'PATIENT_BIRTHDATE': 1, 'DOSAGE': 2, 'BIRTHDATE': 3, 'DIAGNOSIS': 4, 'DRUG': 5, 'PATIENT_NAME': 6, 'PHYSICIAN_NAME': 7, 'FREQUENCY': 8, 'DOB': 9, 'HOSPITAL_NAME': 10, 'AETNA_ID': 11, 'GENDER': 12, 'AMOUNT_PAID': 13}
max_len = 256

# Create the dataset using the NERDataset class
dataset = NERDataset(training_data, tokenizer, label_map, max_len)
dataloader = DataLoader(dataset, batch_size=4)

num_labels = len(label_map) + 1
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=num_labels)

### 2.3 Model training

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

for epoch in range(30):  # Number of epochs
    model.train()
    total_loss = 0
    for batch in tqdm(dataloader):
        input_ids, labels = batch
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch}, Loss: {total_loss / len(dataloader)}")


c:\Users\Prayut Jain\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
  0%|          | 0/20 [00:00<?, ?it/s]

100%|██████████| 20/20 [02:32<00:00,  7.62s/it]


Epoch 0, Loss: 0.7960646167397499


100%|██████████| 20/20 [03:41<00:00, 11.09s/it]


Epoch 1, Loss: 0.48663146495819093


...

Epoch 27, Loss: 0.1306528838351369


100%|██████████| 20/20 [02:35<00:00,  7.78s/it]


Epoch 28, Loss: 0.1342194058932364


100%|██████████| 20/20 [03:07<00:00,  9.36s/it]

Epoch 29, Loss: 0.13035095119848847


In [ ]:
model.save_pretrained

### 2.4 Model predictions

In [3]:
model_path = "./ner_model"
model = BertForTokenClassification.from_pretrained(model_path)
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')

In [22]:
make_predictions(model, tokenizer, "Patient's Aetna ID Number: 66397571968696, Patient's Name: Linda Williams, \
                 Patient's Birthdate (MM/DD/YYYY): 06/21/2003, Name & address of facility where services rendered \
                 (if other than home or office): Mercy Hospital, 2525 S Michigan Ave, Chicago, IL 60616, Diagnosis \
                 or nature of illness or injury (please indicate primary and secondary): Primary Diagnosis: Depression, \
                 Diagnosis or nature of illness or injury (please indicate primary and secondary): Secondary Diagnosis: \
                 Gastroesophageal Reflux Disease (GERD), Description of Service: Fluoxetine, 20mg, Once daily, Description \
                 of Service: Omeprazole, 40mg, Once daily before breakfast, Description of Service: Alprazolam, 0.25mg, \
                 As needed for anxiety, Physician's Name & Address (include ZIP Code): Dr. Johnathan Smith\n1234 Elm Street\
                 \nSuite 200\nChicago, IL 60601, Amount paid $: 8683")

'{"AETNA_ID": "66397571968696", "PATIENT_NAME": "s name linda williams", "PATIENT_BIRTHDATE": "/ dd / yyyy )", "HOSPITAL_NAME": "home or office )", "DIAGNOSIS": "or or injury ( please indicate primary and )", "DRUG": "of", "FREQUENCY": "omeprazole 40mg"}'

## Step 3: Summarisation

### 3.0 Extraction using our model

In [122]:
annotator = annotate_data()

input_pdf_path = "generated_forms\cpdcf_45.pdf"
field_names = annotator.extract_readable_form_field_names(input_pdf_path)
keys_to_extract = ['3. Text.1', '3. Text.2', '3. CheckBox.1a','3. CheckBox.1cl','Text1.0']
extracted_values = [field_names[key] for key in keys_to_extract if key in field_names]
input_text = ', '.join([f'{pair[0]}: {pair[1]}' for pair in extracted_values])

In [123]:
json_str = make_predictions(model, tokenizer, input_text)
data = json.loads(json_str)

# Extract the desired information
aetna = data.get("AETNA_ID", "Not specified")
name = data.get("PATIENT_NAME", "Not specified")
dob = data.get("PATIENT_BIRTHDATE", "Not specified")

print(f"AETNA: {aetna}, Name: {name}, DOB: {dob}")

AETNA: ) 69878216699540, Name: mary johnson, DOB: ) 08 / 20 /


In [48]:
input_pdf_path = "generated_forms\mbr_79.pdf"
field_names = annotator.extract_readable_form_field_names(input_pdf_path)
keys_to_extract = ['Text16', 'Text15', 'Text17','Text63','Text64','Text65','Text71','Text79','Text87','Text100','Text106']
extracted_values = [field_names[key] for key in keys_to_extract if key in field_names]
input_text = ', '.join([f'{pair[0]}: {pair[1]}' for pair in extracted_values])

In [50]:
json_str = make_predictions(model, tokenizer, input_text)
data = json.loads(json_str)

# Extract the desired information
aetna = data.get("AETNA_ID", "Not specified")
drug = data.get("DRUG", "Not specified")
dosage = data.get("DOSAGE", "Not specified")
frequency = data.get("FREQUENCY", "Not specified")

print(f"AETNA: {aetna}, Drug: {drug}, Dosage: {dosage}, Frequency: {frequency}")

AETNA: number 60397228599800, Drug: propranolol, Dosage: sumatriptan, Frequency: 80mg twice daily


### 3.1 Extraction and summarisation using pretrained clinical model

In [3]:
annotator = annotate_data()
input_pdf_path = "generated_forms\mbr_79.pdf"
field_names = annotator.extract_readable_form_field_names(input_pdf_path)
cleaned_dict = {k: v for k, v in field_names.items() if None not in v}

In [11]:
target_words = ['diagnosis', 'drug', 'dosage', 'service']
# Load the spaCy model
nlp = spacy.load("en_core_web_sm")

selected_spacy = []
for key, value in cleaned_dict.items():
    text = str(value[0])
    doc = nlp(text)
    if any(token.text.lower() in target_words for token in doc):
        selected_spacy.append(str(value[1]))

selected_spacy

['Primary Diagnosis: High Cholesterol',
 'Secondary Diagnosis: Migraine',
 '08/15/2023',
 'Atorvastatin, 20mg, Once daily',
 '08/15/2023',
 'Propranolol, 80mg, Twice daily',
 '08/15/2023',
 'Sumatriptan, 100mg, As needed for migraines']

In [24]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

model = AutoModelForTokenClassification.from_pretrained("Posos/ClinicalNER")
tokenizer = AutoTokenizer.from_pretrained("Posos/ClinicalNER")


c:\Users\Prayut Jain\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:133: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Prayut Jain\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [39]:
structured_summary = summarise_results(model, tokenizer, selected_spacy)
structured_summary

{'Diagnosis': ['High Cholesterol', 'Migraine'],
 'Prescriptions': [{'Drug': '<s>▁Atorvastatin</s>',
   'Dosage': '▁20mg',
   'Frequency': '▁Once▁daily'},
  {'Drug': '▁Propranolol', 'Dosage': '▁80mg', 'Frequency': '▁Twice▁daily'},
  {'Drug': '▁Sumatriptan', 'Dosage': '▁100mg', 'Frequency': '▁As▁needed'}]}